In [ ]:
import sys
import torch
import functools
import matplotlib.pyplot as plt
import argparse, yaml, os
import torch.optim.lr_scheduler as lr_scheduler
import torchvision.transforms as transforms
import seaborn as sns
import pandas as pd
import tqdm
import glob
import json

from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from types import SimpleNamespace

from IPython.display import clear_output

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.statistics_sets import (
    STAT_SET_FULL_MCDERMOTTSIMONCELLI as statistics_dict
)
from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params as model_params_tm
from texture_prior.params import statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")
import DistanceMemoryModelScheduledNoise
import encoders
import ScoreFunction
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability

from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.loading import load_results_with_exclusion_2
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utls.reliability import compute_power_curve
from utls.plotting import plot_power_curve


def get_dprime_by_isi(df_per_subject, return_sem=False, return_subjects=False):
    """
    Compute mean d-prime per ISI across subjects, excluding ISI = -1 (lures).

    Args:
        df_per_subject (pd.DataFrame): Output from recompute_dprime_by_isi_per_subject.
        return_sem (bool): Whether to return standard error of the mean.
        return_subjects (bool): Whether to return per-subject d-primes too.

    Returns:
        pd.DataFrame or dict:
            If return_sem=False:
                DataFrame with columns ['isi', 'd_prime']
            If return_sem=True:
                DataFrame with columns ['isi', 'd_prime', 'sem']
            If return_subjects=True:
                Returns a dict with:
                    'summary': summary DataFrame as above,
                    'per_subject': filtered per-subject df
    """
    df_filtered = df_per_subject[df_per_subject["isi"] > -1]

    grouped = df_filtered.groupby("isi")["d_prime"]
    result_df = grouped.mean().reset_index(name="d_prime")

    if return_sem:
        result_df["sem"] = grouped.sem().values

    if return_subjects:
        return {
            "summary": result_df,
            "per_subject": df_filtered.copy()
        }

    return result_df.d_prime.tolist()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# grabbing example list of sound
sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")
texture_list = sounds_list

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")
print(len(ALL_SOUNDS))

tasks = ["ind-nature-len120" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
which_task = tasks[2] # "global-music-len120", "atexts-len120" "nhs-region-len120"

base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"

seqs_paths = {"ind-nature-len120": "mem_exp_ind-nature_2025", 
              "global-music-len120": "global-music-2025-n_80",
              "atexts-len120": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}

hr_task_name = {"ind-nature-len120": "Industrial and Nature", 
              "global-music-len120": "Globalized Music",
              "atexts-len120": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

exps, seqs, fnames = load_results(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}")
move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)


In [ ]:
device = "cuda"
pc_dims = 256

pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)

zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=device)
zscore_projector.fit(sounds_list)

sf = ScoreFunction.ScoreFunction(device = device)

In [ ]:
skip = False

if skip: 
    
    exps, seqs, fnames, _ = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                        min_dprime=2,
                                                        min_trials=120,
                                                        skip_len60=True,
                                                        verbose=False,
                                                        return_skipped=skip)
else:
    exps, seqs, fnames = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                    min_dprime=2,
                                                    min_trials=120,
                                                    skip_len60=True,
                                                    verbose=False,
                                                    return_skipped=skip)



move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

print("Number of participants used in analysis:", len(exps))


safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only", safe_name)

# where you want the results
CSV_PATH = f"/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/"

ensure_dir(save_dir)
ensure_dir(CSV_PATH)
print(save_dir)

human_results = recompute_dprime_by_isi_per_subject(exps)
human_sensitivity = get_dprime_by_isi(human_results)

clear_output(wait=False)
CSV_PATH = f"/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/isi16_runs.csv"

from sklearn.metrics import mean_squared_error

# Your experiment structure (list of stimulus filepaths for each run)
experiment_list = []
for seq in seqs:
    list_to_add = []
    
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as file:
        data = json.load(file)
   
    for stim in  data['filenames_order']:
        edited_stim_name = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + stim
        list_to_add.append(edited_stim_name)
    
    experiment_list.append(list_to_add)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict

# ----------------------------- CONFIG ------------------------------------
NV = 0.2
CRIT_MULT = 1.5
SEED = 123
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TIMES_TO_PLOT = [10, 17, 16, 22, 40, 80, 119]

if SEED is not None:
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

# ------------------- ENCODE CLEAN REPRESENTATIONS ------------------------
seen, all_files = set(), []
for seq in experiment_list:
    for fn in seq:
        if fn not in seen:
            seen.add(fn)
            all_files.append(fn)

_tmp = DistanceMemoryModelScheduledNoise.DistanceMemoryModelScheduledNoise(
    encoding_model=zscore_projector,
    max_variance=float(NV),
    min_variance=0.0001,
    criterion=0.0,
    device=DEVICE
)
_tmp._fill_memory_bank(all_files)
with torch.no_grad():
    X0 = torch.stack([rep.detach().clone().view(-1) for rep in _tmp.memory_bank], dim=0).to(DEVICE)
del _tmp
if torch.cuda.is_available():
    torch.cuda.empty_cache()

M, D = X0.shape
crit = float(NV * math.sqrt(16*D))  # pick a representative ISI=16 threshold
name_to_idx = {fn: i for i, fn in enumerate(all_files)}

# --------------------- STORAGE -------------------------------------------
T_max = max((len(seq) for seq in experiment_list), default=0)
hit_dists_by_t = [[] for _ in range(T_max)]
fa_dists_by_t  = [[] for _ in range(T_max)]
hit_yes_num = np.zeros(T_max, dtype=float)
hit_yes_den = np.zeros(T_max, dtype=float)
fa_yes_num  = np.zeros(T_max, dtype=float)
fa_yes_den  = np.zeros(T_max, dtype=float)

isi_hit_dists = defaultdict(list)
isi_yes_num   = defaultdict(float)
isi_yes_den   = defaultdict(float)

# --------------------------- MAIN LOOP -----------------------------------
for seq in experiment_list:
    if len(seq) == 0:
        continue

    model = DistanceMemoryModelScheduledNoise.DistanceMemoryModelScheduledNoise(
        encoding_model=zscore_projector,
        max_variance=float(NV),
        min_variance=0.0001,
        total_steps=32,
        criterion=float(crit),
        device=DEVICE
    )
    model.memory_bank = []
    model.ts = []

    seq_idx = [name_to_idx[fn] for fn in seq]
    bank_set, last_seen = set(), {}

    with torch.no_grad():
        for t, idx_incoming in enumerate(seq_idx, start=1):
            # Drift existing bank (applies per-trace schedule)
            if model.memory_bank:
                model.apply_noise_to_memory()

            # Measure distance BEFORE inserting
            if model.memory_bank:
                bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                probe = X0[idx_incoming].view(1, -1)
                dmin = torch.linalg.norm(bank_mat - probe, dim=1).min().item()

                if idx_incoming in bank_set:
                    hit_dists_by_t[t-1].append(dmin)
                    hit_yes_num[t-1] += float(dmin < crit)
                    hit_yes_den[t-1] += 1.0

                    isi = t - last_seen[idx_incoming]
                    isi_hit_dists[isi].append(dmin)
                    isi_yes_num[isi] += float(dmin < crit)
                    isi_yes_den[isi] += 1.0
                else:
                    fa_dists_by_t[t-1].append(dmin)
                    fa_yes_num[t-1]  += float(dmin < crit)
                    fa_yes_den[t-1]  += 1.0

            # Insert probe (clean rep)
            model.memory_bank.append(X0[idx_incoming].detach().clone())
            model.ts.append(0)
            model.ts = [age+1 for age in model.ts]
            bank_set.add(idx_incoming)
            last_seen[idx_incoming] = t

# ------------------------ REDUCE & PLOT ----------------------------------
hit_yes_rate = np.divide(hit_yes_num, np.maximum(hit_yes_den, 1.0))
fa_yes_rate  = np.divide(fa_yes_num,  np.maximum(fa_yes_den,  1.0))

# Time course plots
x = np.arange(1, T_max+1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
axes[0].plot(x, [np.mean(h) if h else np.nan for h in hit_dists_by_t], color='green', label='hit min dist')
axes[0].plot(x, [np.mean(f) if f else np.nan for f in fa_dists_by_t],  color='red',   label='FA min dist')
axes[0].axhline(crit, linestyle='--', color='k', label=f'criterion={crit:.3g}')
axes[0].set_title('Nearest distance over insertion steps')
axes[0].set_xlabel('step')
axes[0].set_ylabel('min distance')
axes[0].legend()

axes[1].plot(x, hit_yes_rate, color='green', label='P(yes|hit)')
axes[1].plot(x, fa_yes_rate,  color='red',   label='P(yes|FA)')
axes[1].set_ylim(0, 1)
axes[1].set_title('Decision probabilities')
axes[1].legend()
plt.tight_layout()
plt.show()

# ISI plots
if len(isi_hit_dists) > 0:
    isis = sorted(isi_hit_dists.keys())
    isi_mean = [np.mean(isi_hit_dists[k]) for k in isis]
    isi_sem  = [np.std(isi_hit_dists[k], ddof=1)/np.sqrt(len(isi_hit_dists[k])) if len(isi_hit_dists[k])>1 else np.nan for k in isis]
    isi_pyes = [isi_yes_num[k]/max(isi_yes_den[k],1.0) for k in isis]

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].errorbar(isis, isi_mean, yerr=isi_sem, fmt='o-')
    ax[0].axhline(crit, linestyle='--', color='k')
    ax[0].set_title('Hit min distance vs ISI')
    ax[0].set_xlabel('ISI (trials)')
    ax[0].set_ylabel('min distance')

    ax[1].plot(isis, isi_pyes, 'o-')
    ax[1].set_ylim(0, 1)
    ax[1].set_title('Hit decision probability vs ISI')
    ax[1].set_xlabel('ISI (trials)')
    ax[1].set_ylabel('P(yes)')
    plt.tight_layout()
    plt.show()

# B) Histograms at selected steps (overlay hit vs FA). Use a common x-range via 99.5th percentile.
valid_steps = [t for t in TIMES_TO_PLOT if 1 <= t <= T_max]
if len(valid_steps) > 0:
    pooled = []
    for t in valid_steps:
        pooled.extend(hit_dists_by_t[t-1])
        pooled.extend(fa_dists_by_t[t-1])
    pooled = np.asarray(pooled, dtype=float)
    xmax = float(np.percentile(pooled, 99.5)) if pooled.size else 1.0

    cols = len(valid_steps)
    fig2, axes2 = plt.subplots(1, cols, figsize=(4.4 * cols, 3.6), sharey=True)
    if cols == 1:
        axes2 = [axes2]

    for ax, t in zip(axes2, valid_steps):
        h = np.asarray(hit_dists_by_t[t-1], dtype=float)
        f = np.asarray(fa_dists_by_t[t-1],  dtype=float)
        ax.hist(h, bins=40, range=(0, xmax), density=True, alpha=0.55, color='green', label='in bank')
        ax.hist(f, bins=40, range=(0, xmax), density=True, alpha=0.55, color='red',   label='out of bank')
        ax.axvline(float(NV * math.sqrt(17*D)), linestyle='--', linewidth=2, color='k', label=f'criterion = {float(NV * math.sqrt(17*D)):.3g}')
        ax.axvline(float(NV * math.sqrt(1*D)), linestyle='-', linewidth=2, color='k', label=f'criterion = {float(NV * math.sqrt(1*D)):.3g}')

        py_h = float(np.mean(h < crit)) if h.size else 0.0
        py_f = float(np.mean(f < crit)) if f.size else 0.0
        ax.set_title(f't={t} | Pyes(hit)={py_h:.2f} | Pyes(FA)={py_f:.2f}')
        ax.set_xlabel('min distance')
        if t == valid_steps[0]:
            ax.set_ylabel('density')
        ax.legend(fontsize=8, loc='upper right')

    fig2.suptitle('Min-distance distributions by step (actual sequences)', y=0.98)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.hist(isi_hit_dists[1], bins=100, color='b', label="ISI0 Min. distances", alpha=0.5)
plt.hist(isi_hit_dists[17], bins=100, color='m', label="ISI16 Min. distances", alpha=0.5)

plt.axvline(np.mean(isi_hit_dists[1]), color='b', linestyle='-', label="Empirical ISI0 Min. distances")
plt.axvline(np.mean(isi_hit_dists[17]), color='m', linestyle='-', label="Empirical ISI16 Min. distances")

plt.axvline(float(NV * math.sqrt(1*D)), linestyle='--', color='b',  label="Theoretical ISI0 Min. distances")
plt.axvline(float(NV * math.sqrt(17*D)), linestyle='--', color='m', label="Theoretical ISI16 Min. distances")

plt.axvline(np.mean(isi_hit_dists[1] + isi_hit_dists[17]), linestyle=':', color='b',  label="Empirical Avg. Dist")
avg_isi_dist = float(NV * math.sqrt(1*D))*(10/40) + float(NV * math.sqrt(17*D))*(30/40)

plt.axvline(avg_isi_dist, linestyle=":", color='m', label="Theoretical Avg. Dist")

plt.legend()

In [ ]:
#check the sound level analysis

In [ ]:
# # Optional: itemwise mean distance vs ISI line plots for selected items
# # Example: plot the top 6 most frequent items across ISIs
# counts = (item_isi_df.groupby("filename")["n_trials"].sum()
#           .sort_values(ascending=False))
# top_items = list(counts.head(80).index)
# if len(top_items) > 0:
#     plt.figure(figsize=(7.5, 4.6))
#     for nm in top_items:
#         sub = item_isi_df[item_isi_df["filename"] == nm].sort_values("ISI")
#         plt.errorbar(sub["ISI"], sub["mean_dmin"], yerr=sub["sem_dmin"], marker="o", label=nm, alpha=0.2)
#     plt.axhline(crit, linestyle="--", color="k", linewidth=2, label=f"criterion = {crit:.3g}")
#     plt.xlabel("ISI")
#     plt.ylabel("hit min distance (mean ± SEM)")
#     plt.title("Per-item hit min distance vs ISI (examples)")
#     #plt.legend(fontsize=8, ncol=2)
#     plt.tight_layout()
#     plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Flatten lists
hit_dists = np.concatenate([np.array(x) for x in hit_dists_by_t if len(x) > 0])
fa_dists  = np.concatenate([np.array(x) for x in fa_dists_by_t  if len(x) > 0])

# Labels: 1 for hits (positives), 0 for FAs (negatives)
y_true = np.concatenate([np.ones_like(hit_dists), np.zeros_like(fa_dists)])

# Scores: smaller distance = more likely "yes"
y_score = np.concatenate([hit_dists, fa_dists])
y_score = -y_score   # flip sign so higher = more match-like

# ROC
fpr, tpr, thresholds = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

# Plot
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC from min distances")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

def flatten_dists(hit_lists, fa_lists):
    """Flatten lists of distances for hits and FAs into (labels, scores)."""
    hits = np.concatenate([np.array(x, dtype=float) for x in hit_lists if len(x) > 0])
    fas  = np.concatenate([np.array(x, dtype=float) for x in fa_lists  if len(x) > 0])
    if hits.size == 0 or fas.size == 0:
        return None, None
    y_true  = np.concatenate([np.ones_like(hits), np.zeros_like(fas)])
    y_score = np.concatenate([hits, fas])
    return y_true, -y_score   # negate: smaller dmin = more match-like

def compute_roc(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return fpr, tpr, auc(fpr, tpr)

# --- Full ---
y_true_full, y_score_full = flatten_dists(hit_dists_by_t, fa_dists_by_t)
fpr_full, tpr_full, auc_full = compute_roc(y_true_full, y_score_full)

# --- First half ---
half_point = T_max // 2
y_true_1st, y_score_1st = flatten_dists(hit_dists_by_t[:half_point], fa_dists_by_t[:half_point])
fpr_1st, tpr_1st, auc_1st = compute_roc(y_true_1st, y_score_1st)

# --- Second half ---
y_true_2nd, y_score_2nd = flatten_dists(hit_dists_by_t[half_point:], fa_dists_by_t[half_point:])
fpr_2nd, tpr_2nd, auc_2nd = compute_roc(y_true_2nd, y_score_2nd)

# --- Plot overlay ---
plt.figure(figsize=(6,6))
plt.plot(fpr_full, tpr_full, label=f'Full (AUC={auc_full:.2f})', color='black')
plt.plot(fpr_1st, tpr_1st, label=f'1st half (AUC={auc_1st:.2f})', color='blue')
plt.plot(fpr_2nd, tpr_2nd, label=f'2nd half (AUC={auc_2nd:.2f})', color='red')
plt.plot([0,1], [0,1], 'k--', alpha=0.5)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves: full vs halves")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

def roc_from_isi(isi_list, fa_dists_by_t, isi_value):
    """Build ROC for a given ISI value using pooled FA dists as negatives."""
    hits = np.array(isi_hit_dists.get(isi_value, []), dtype=float)
    fas  = np.concatenate([np.array(x) for x in fa_dists_by_t if len(x) > 0])

    if hits.size == 0 or fas.size == 0:
        return None, None, None, None

    y_true  = np.concatenate([np.ones_like(hits), np.zeros_like(fas)])
    y_score = np.concatenate([hits, fas])
    y_score = -y_score  # smaller distance = more match-like

    fpr, tpr, thr = roc_curve(y_true, y_score)
    return fpr, tpr, thr, auc(fpr, tpr)

import numpy as np
from sklearn.metrics import roc_curve, auc

def average_rocs(roc_dict, method="macro", n_points=200):
    """
    Average ROC curves across conditions.
    roc_dict : {cond: (fpr, tpr, auc_val)} for each ISI
    method   : "macro" (average TPR at fixed FPR grid) or "micro" (pool data)
    """
    if method == "macro":
        # Common FPR grid
        fpr_grid = np.linspace(0, 1, n_points)
        tpr_interp = []
        for fpr, tpr, _ in roc_dict.values():
            tpr_i = np.interp(fpr_grid, fpr, tpr)
            tpr_interp.append(tpr_i)
        mean_tpr = np.mean(tpr_interp, axis=0)
        mean_auc = auc(fpr_grid, mean_tpr)
        return fpr_grid, mean_tpr, mean_auc
    
    elif method == "micro":
        # Pool all points into one ROC (requires re-deriving labels/scores)
        raise NotImplementedError("Micro-average needs raw labels & scores pooled.")

# Example: ROC for ISI=0 and ISI=16
roc_curves = {}
for isi in [1, 17]:
    fpr, tpr, thr, auc_val = roc_from_isi(isi_hit_dists, fa_dists_by_t, isi)
    if fpr is not None:
        roc_curves[isi] = (fpr, tpr, auc_val)

# Plot overlay
plt.figure(figsize=(6,6))
for isi, (fpr, tpr, auc_val) in roc_curves.items():
    plt.plot(fpr, tpr, label=f"ISI={isi-1}, AUC={auc_val:.2f}")
plt.plot([0,1], [0,1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves by ISI condition")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

def run_experiment_at_noise(NV_value):
    """Run memory bank simulation at one noise level and collect distances."""
    hit_dists, fa_dists = [], []

    for seq in experiment_list:
        if len(seq) == 0:
            continue

        # fresh model
        model = DistanceMemoryPriorModel.DistanceMemoryPriorModel(
            encoding_model=zscore_projector,
            drift_step_size=0.001,
            score_model = sf, 
            noise_variance=float(NV_value),
            criterion=0.0,   # criterion not needed for distance collection
            device=DEVICE
        )
        model.memory_bank = []

        seq_idx = [name_to_idx[fn] for fn in seq]

        bank_set, last_seen = set(), {}

        with torch.no_grad():
            for t, idx_incoming in enumerate(seq_idx, start=1):
                if len(model.memory_bank) > 0:
                    model.apply_noise_to_memory()

                if len(model.memory_bank) > 0:
                    bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                    probe = X0[idx_incoming].view(1, -1)
                    dmin = torch.linalg.norm(bank_mat - probe, dim=1).min().item()

                    if idx_incoming in bank_set:
                        hit_dists.append(dmin)
                    else:
                        fa_dists.append(dmin)

                model.memory_bank.append(X0[idx_incoming].detach().clone())
                bank_set.add(idx_incoming)
                last_seen[idx_incoming] = t

    return np.array(hit_dists), np.array(fa_dists)


def roc_from_dists(hit_dists, fa_dists):
    """Compute ROC from hit and FA distance arrays."""
    if hit_dists.size == 0 or fa_dists.size == 0:
        return None, None, None
    y_true  = np.concatenate([np.ones_like(hit_dists), np.zeros_like(fa_dists)])
    y_score = np.concatenate([hit_dists, fa_dists])
    y_score = -y_score  # lower distance = more match-like
    fpr, tpr, thr = roc_curve(y_true, y_score)
    auc_val = auc(fpr, tpr)
    return fpr, tpr, auc_val


# ---------------- Run for a few noise levels ----------------
noise_levels = [0.1, 0.2, 0.3, 0.5, 0.8]   # choose which NVs you want
roc_curves = {}

for nv in noise_levels:
    hits, fas = run_experiment_at_noise(nv)
    res = roc_from_dists(hits, fas)
    if res[0] is not None:
        fpr, tpr, auc_val = res
        roc_curves[nv] = (fpr, tpr, auc_val)

# ---------------- Plot overlay ----------------
plt.figure(figsize=(6,6))
for nv, (fpr, tpr, auc_val) in roc_curves.items():
    plt.plot(fpr, tpr, label=f"noise={nv} | AUC={auc_val:.2f}")
plt.plot([0,1], [0,1], 'k--', alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves across noise levels")
plt.legend()
plt.show()

In [ ]:
# ===================== imports & small utils =====================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from collections import defaultdict

# ===================== core helpers =====================
def roc_from_arrays(hit_dists: np.ndarray, fa_dists: np.ndarray):
    """Return (fpr, tpr, auc_val) from distance arrays."""
    if hit_dists.size == 0 or fa_dists.size == 0:
        return None
    y_true  = np.concatenate([np.ones_like(hit_dists), np.zeros_like(fa_dists)])
    y_score = np.concatenate([hit_dists, fa_dists])
    y_score = -y_score  # lower dmin = more match-like
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return fpr, tpr, auc(fpr, tpr)

def plot_roc(ax, fpr, tpr, label):
    """Plot a single ROC curve on ax."""
    ax.plot(fpr, tpr, label=label)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# ===================== experiment runner =====================
def run_experiment_at_noise(
    NV_value: float,
    *,
    X0,                            # [M, D] clean embeddings (torch not required here)
    name_to_idx: dict,
    experiment_list,               # list[list[str]]
    DEVICE="cpu",
    DistanceMemoryModel=None,
    zscore_projector=None
):
    """
    Run sequences once at a given noise variance.
    Returns:
      {
        "hits": np.ndarray,             # all hit dmins (pooled)
        "fas":  np.ndarray,             # all FA dmins (pooled)
        "isi_hit_dists": {isi: [(dmin, t), ...]},
        "fa_by_t": list[list[float]],   # index t-1 -> FA dmins at trial t (pooled across sequences)
        "T_max": int                    # longest sequence length
      }
    """
    # per-run accumulators
    hit_dists, fa_dists = [], []
    isi_hit_dists = defaultdict(list)     # isi -> list of (dmin, t)
    T_max = max((len(seq) for seq in experiment_list), default=0)
    fa_by_t = [[] for _ in range(T_max)]

    for seq in experiment_list:
        if not seq:
            continue

        model = DistanceMemoryModel.DistanceMemoryModel(
            encoding_model=zscore_projector,
            noise_variance=float(NV_value),
            criterion=0.0,
            device=DEVICE
        )
        model.memory_bank = []

        seq_idx = [name_to_idx[fn] for fn in seq]
        bank_set, last_seen = set(), {}

        # iterate trials
        import torch  # local import to avoid leaking torch usage elsewhere
        with torch.no_grad():
            for t, idx_incoming in enumerate(seq_idx, start=1):
                if len(model.memory_bank) > 0:
                    model.apply_noise_to_memory()

                if len(model.memory_bank) > 0:
                    bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                    probe = X0[idx_incoming].view(1, -1)
                    dmin = torch.linalg.norm(bank_mat - probe, dim=1).min().item()

                    if idx_incoming in bank_set:
                        hit_dists.append(dmin)
                        isi = t - last_seen[idx_incoming]  # ISI in trials
                        isi_hit_dists[isi].append((dmin, t))
                    else:
                        fa_dists.append(dmin)
                        fa_by_t[t-1].append(dmin)

                # insert clean probe copy
                model.memory_bank.append(X0[idx_incoming].detach().clone())
                bank_set.add(idx_incoming)
                last_seen[idx_incoming] = t

    return {
        "hits": np.asarray(hit_dists, float),
        "fas":  np.asarray(fa_dists,  float),
        "isi_hit_dists": isi_hit_dists,
        "fa_by_t": fa_by_t,
        "T_max": T_max,
    }

# ===================== per-noise ROC (overall) =====================
def rocs_across_noise(noise_levels, *, X0, name_to_idx, experiment_list, DistanceMemoryModel, zscore_projector, DEVICE="cpu"):
    """
    For each NV, run once and compute overall ROC. Also keep per-ISI & per-trial logs for later slicing.
    Returns:
      curves: {nv: (fpr, tpr, auc)}
      runs:   {nv: run_data_dict_from_runner}
    """
    curves, runs = {}, {}
    for nv in noise_levels:
        run = run_experiment_at_noise(
            nv, X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
            DEVICE=DEVICE, DistanceMemoryModel=DistanceMemoryModel, zscore_projector=zscore_projector
        )
        runs[nv] = run
        res = roc_from_arrays(run["hits"], run["fas"])
        if res is not None:
            curves[nv] = res  # (fpr, tpr, auc)
    return curves, runs

# ===================== ISI-specific ROC & second-half ROC =====================
def roc_for_isi(run_data, isi_value: int):
    """ROC using hits at given ISI vs all pooled FAs."""
    hits_raw = run_data["isi_hit_dists"].get(isi_value, [])
    if not hits_raw or run_data["fas"].size == 0:
        return None
    hits = np.array([d for (d, _t) in hits_raw], float)
    return roc_from_arrays(hits, run_data["fas"])

def roc_for_second_half(run_data):
    """ROC using only trials t > T_max//2. FAs filtered by trial index, hits filtered by their logged t."""
    T_half = run_data["T_max"] // 2
    # hits: collect all (d,t) across ISIs, keep those in second half
    hits = []
    for lst in run_data["isi_hit_dists"].values():
        hits.extend([d for (d, t) in lst if t > T_half])
    hits = np.asarray(hits, float)

    # FAs: use fa_by_t buckets to filter by half
    fas = []
    for t_idx, bucket in enumerate(run_data["fa_by_t"], start=1):
        if t_idx > T_half:
            fas.extend(bucket)
    fas = np.asarray(fas, float)

    return roc_from_arrays(hits, fas)

# ===================== plotting wrappers =====================
def plot_noise_overlays(curves_by_noise, title="ROC curves across noise levels"):
    """Overlay overall ROC per noise."""
    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    for nv, (fpr, tpr, auc_val) in curves_by_noise.items():
        plot_roc(ax, fpr, tpr, label=f"noise={nv:g} | AUC={auc_val:.3f}")
    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

def plot_isi_and_half_for_noise(nv: float, run_data, isis=(1,17)):
    """
    For a single noise level: plot ROC for ISI=1, ISI=17, and Second Half.
    """
    fig, ax = plt.subplots(figsize=(6.2, 6.2))

    # ISI curves
    for k in isis:
        res = roc_for_isi(run_data, k)
        if res is not None:
            fpr, tpr, auc_val = res
            plot_roc(ax, fpr, tpr, label=f"ISI={k} | AUC={auc_val:.3f}")

    # Second-half curve
    res_half = roc_for_second_half(run_data)
    if res_half is not None:
        fpr, tpr, auc_val = res_half
        plot_roc(ax, fpr, tpr, label=f"Second half | AUC={auc_val:.3f}")

    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"Noise={nv:g}: ISI (1,17) and Second-Half ROCs")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

# ===================== USAGE EXAMPLE =====================
# Assumes you already have: X0 (torch.Tensor [M,D]), name_to_idx (dict), experiment_list (list[list[str]]),
# DistanceMemoryModel (module), zscore_projector (encoder), DEVICE (str).
# Pick the noise levels you want:
noise_levels = [0.2, 0.3, 0.5, 0.8, 1]

# 1) Overall ROCs across noise
curves_by_noise, runs_by_noise = rocs_across_noise(
    noise_levels,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    DistanceMemoryModel=DistanceMemoryModel, zscore_projector=zscore_projector, DEVICE=DEVICE
)
plot_noise_overlays(curves_by_noise, title="ROC across noise levels (overall)")

# 2) For a few selected noise levels, plot ISI=1, ISI=17, and Second Half
for nv_focus in noise_levels:  # choose any subset to inspect
    if nv_focus in runs_by_noise:
        plot_isi_and_half_for_noise(nv_focus, runs_by_noise[nv_focus], isis=(1, 17))